# 03. Validacion de la base clima-satelite mensual

Este notebook revisa la base mensual generada a partir de `PT1` para confirmar si puede usarse como insumo confiable en la integracion posterior con Agronet.

## Objetivos
- cargar la base clima-satelite de entrada
- validar estructura, cobertura temporal y granularidad
- revisar duplicados por `fecha-departamento`
- revisar nulos por variable y por periodo
- verificar consistencia basica de variables climaticas, satelitales y de terreno
- dejar una copia validada en `BASE_DE_DATOS/INTERMEDIAS`

## Resultado esperado
Al finalizar este notebook debemos saber:

1. si la base tiene exactamente una fila por `departamento-mes`
2. si la cobertura temporal es completa y coherente
3. si los nulos se concentran al final de la serie o si hay huecos estructurales
4. si las variables clave tienen rangos razonables
5. si la base puede pasar al siguiente notebook sin regenerarse desde cero


## Comentario metodologico

Este notebook no corrige todavia la base. Primero la audita. Si encontramos un problema grave, la idea es dejarlo explicitamente documentado antes de integrar o imputar.

La salida de este notebook sera una copia validada de trabajo, no una version "limpia" definitiva. La limpieza operacional y la integracion con otras capas se hara en notebooks posteriores.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.max_colwidth', 140)


def find_project_root(start: Path) -> Path:
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / 'BASE_DE_DATOS').exists():
            return candidate
    raise FileNotFoundError('No se encontro una carpeta BASE_DE_DATOS en la ruta actual ni en sus padres.')


CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = find_project_root(CURRENT_DIR)
BASE_DATOS = PROJECT_ROOT / 'BASE_DE_DATOS'
INPUT_PATH = BASE_DATOS / 'ENTRADA' / 'datos_seguro_cafe_2000_2025.csv'
OUTPUT_DIR = BASE_DATOS / 'INTERMEDIAS'
OUTPUT_PATH = OUTPUT_DIR / 'clima_satelite_mensual_validada.csv'

print(f'Ruta actual: {CURRENT_DIR}')
print(f'Raiz detectada del proyecto: {PROJECT_ROOT}')
print(f'Archivo de entrada: {INPUT_PATH}')
print(f'Archivo de salida: {OUTPUT_PATH}')


In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f'No existe el archivo esperado: {INPUT_PATH}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Rutas validadas correctamente.')


## Carga de la base y estandarizacion minima

La base original viene como CSV separado por comas. Aqui se carga, se parsea la fecha y se ordena para facilitar las validaciones posteriores.


In [ ]:
df = pd.read_csv(INPUT_PATH)
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
df = df.sort_values(['departamento', 'fecha']).reset_index(drop=True)

print('Shape inicial:', df.shape)
print('Columnas:', df.columns.tolist())
display(df.head(5))


## Perfil general de la base

Primero revisamos la estructura esperada: periodo, departamentos, columnas y granularidad.


In [ ]:
perfil_general = {
    'shape': df.shape,
    'n_columnas': len(df.columns),
    'fecha_min': str(df['fecha'].min()),
    'fecha_max': str(df['fecha'].max()),
    'departamentos': sorted(df['departamento'].dropna().astype(str).unique().tolist()),
    'anios_min_max': (int(df['anio'].min()), int(df['anio'].max())),
    'meses_unicos': sorted(pd.to_numeric(df['mes'], errors='coerce').dropna().astype(int).unique().tolist()),
}

print(json.dumps(perfil_general, indent=2, ensure_ascii=False))


## Cobertura esperada del panel `departamento-mes`

Como esta base es mensual y solo cubre dos departamentos, esperamos una grilla completa sin huecos estructurales entre la fecha minima y maxima.


In [ ]:
departamentos = sorted(df['departamento'].dropna().unique().tolist())
fechas_esperadas = pd.date_range(df['fecha'].min(), df['fecha'].max(), freq='MS')
panel_esperado = pd.MultiIndex.from_product([departamentos, fechas_esperadas], names=['departamento', 'fecha'])
panel_observado = pd.MultiIndex.from_frame(df[['departamento', 'fecha']])

faltantes_panel = panel_esperado.difference(panel_observado)
sobrantes_panel = panel_observado.difference(panel_esperado)

resumen_panel = {
    'n_filas_observadas': len(df),
    'n_filas_esperadas': len(panel_esperado),
    'n_faltantes_panel': len(faltantes_panel),
    'n_sobrantes_panel': len(sobrantes_panel),
}

print(json.dumps(resumen_panel, indent=2, ensure_ascii=False))


In [ ]:
if len(faltantes_panel) > 0:
    display(pd.DataFrame(list(faltantes_panel), columns=['departamento', 'fecha']).head(20))
else:
    print('No hay huecos estructurales en la grilla departamento-mes.')


## Duplicados

La base debe tener como maximo una fila por `fecha-departamento`. Aqui revisamos duplicados totales y duplicados por llave natural.


In [ ]:
duplicados_totales = int(df.duplicated().sum())
duplicados_llave = int(df.duplicated(subset=['fecha', 'departamento']).sum())

print('Duplicados totales:', duplicados_totales)
print('Duplicados por fecha-departamento:', duplicados_llave)


## Clasificacion de variables por grupos

Agrupamos las columnas para revisar nulos y consistencia de forma mas interpretable.


In [ ]:
id_cols = ['fecha', 'departamento', 'fecha_dt', 'mes', 'anio']
clima_cols = ['precipitation', 'temp_aire_C', 'humedad_relativa_pct', 'soil', 'def', 'pet', 'aet', 'GDD_cafe']
sat_cols = ['NDVI', 'EVI', 'LST_Day_1km', 'LST_Night_1km', 'Gpp', 'Lai_500m', 'Fpar_500m']
terrain_cols = ['elevacion_media_m', 'elevacion_std_m', 'pendiente_media', 'pendiente_std', 'aspecto_medio']
anom_cols = ['NDVI_anomalia_pct', 'EVI_anomalia_pct', 'Gpp_anomalia_pct', 'Lai_500m_anomalia_pct', 'precipitation_anomalia_pct']
indice_cols = ['indice_perdida', 'evento_perdida']

group_map = {
    'identificacion': [c for c in id_cols if c in df.columns],
    'clima': [c for c in clima_cols if c in df.columns],
    'satelite': [c for c in sat_cols if c in df.columns],
    'terreno': [c for c in terrain_cols if c in df.columns],
    'anomalias': [c for c in anom_cols if c in df.columns],
    'indice': [c for c in indice_cols if c in df.columns],
}

group_map


## Nulos por variable

Aqui revisamos la magnitud de nulos por columna. Despues veremos si esos nulos estan concentrados al final de la serie, lo cual seria esperable en algunas fuentes satelitales o climaticas recientes.


In [ ]:
nulos_por_columna = (
    df.isnull().sum()
    .sort_values(ascending=False)
    .rename('n_nulos')
    .reset_index()
    .rename(columns={'index': 'columna'})
)
nulos_por_columna['pct_nulos'] = nulos_por_columna['n_nulos'] / len(df)
display(nulos_por_columna.head(20))


In [ ]:
resumen_nulos_grupo = []
for grupo, columnas in group_map.items():
    if not columnas:
        continue
    resumen_nulos_grupo.append({
        'grupo': grupo,
        'n_columnas': len(columnas),
        'nulos_totales': int(df[columnas].isnull().sum().sum()),
        'promedio_nulos_por_columna': float(df[columnas].isnull().sum().mean()),
    })

display(pd.DataFrame(resumen_nulos_grupo))


## Ubicacion temporal de los nulos

No basta con saber cuántos nulos hay. Importa saber si estan concentrados en meses recientes o si hay huecos dispersos dentro de la serie historica.


In [ ]:
def non_null_window(df_in, columns):
    rows = []
    for col in columns:
        serie = df_in[['fecha', col]].dropna(subset=[col]).sort_values('fecha')
        rows.append({
            'columna': col,
            'primera_fecha_no_nula': str(serie['fecha'].min()) if not serie.empty else None,
            'ultima_fecha_no_nula': str(serie['fecha'].max()) if not serie.empty else None,
            'n_no_nulos': int(serie.shape[0]),
        })
    return pd.DataFrame(rows)


ventanas_no_nulas = non_null_window(df, [c for c in df.columns if c not in ['fecha', 'fecha_dt']])
display(ventanas_no_nulas.sort_values(['ultima_fecha_no_nula', 'columna']).head(20))
display(ventanas_no_nulas.sort_values(['ultima_fecha_no_nula', 'columna']).tail(20))


## Nulos por año para variables clave

Este chequeo ayuda a distinguir si los faltantes son un problema historico o si se concentran en el tramo final de la serie.


In [ ]:
vars_clave_nulos = ['soil', 'def', 'pet', 'aet', 'GDD_cafe', 'NDVI', 'EVI', 'LST_Day_1km', 'Gpp', 'Lai_500m', 'Fpar_500m']
vars_clave_nulos = [c for c in vars_clave_nulos if c in df.columns]

nulos_por_anio = df.groupby('anio')[vars_clave_nulos].apply(lambda x: x.isnull().sum())
display(nulos_por_anio.tail(10))


## Rangos y consistencia basica de variables clave

No buscamos validar agronomicamente cada valor puntual, pero si detectar valores imposibles o muy sospechosos para una primera barrida de calidad.


In [ ]:
variables_rangos = [
    'precipitation', 'temp_aire_C', 'humedad_relativa_pct', 'soil', 'def', 'pet', 'aet', 'GDD_cafe',
    'NDVI', 'EVI', 'LST_Day_1km', 'LST_Night_1km', 'Gpp', 'Lai_500m', 'Fpar_500m',
    'NDVI_anomalia_pct', 'EVI_anomalia_pct', 'Gpp_anomalia_pct', 'Lai_500m_anomalia_pct', 'precipitation_anomalia_pct'
]
variables_rangos = [c for c in variables_rangos if c in df.columns]

rangos = df[variables_rangos].agg(['min', 'max', 'mean']).T.reset_index().rename(columns={'index': 'variable'})
display(rangos)


In [ ]:
alertas_rango = []

if 'precipitation' in df.columns:
    alertas_rango.append({'chequeo': 'precipitation_negativa', 'n_filas': int((df['precipitation'] < 0).sum())})

if 'humedad_relativa_pct' in df.columns:
    alertas_rango.append({'chequeo': 'humedad_fuera_0_100', 'n_filas': int(((df['humedad_relativa_pct'] < 0) | (df['humedad_relativa_pct'] > 100)).sum())})

if 'GDD_cafe' in df.columns:
    alertas_rango.append({'chequeo': 'gdd_negativo', 'n_filas': int((df['GDD_cafe'] < 0).sum())})

if 'NDVI' in df.columns:
    alertas_rango.append({'chequeo': 'ndvi_fuera_0_1', 'n_filas': int(((df['NDVI'] < 0) | (df['NDVI'] > 1)).sum())})

if 'EVI' in df.columns:
    alertas_rango.append({'chequeo': 'evi_fuera_rango_amplio', 'n_filas': int(((df['EVI'] < -1) | (df['EVI'] > 1)).sum())})

if 'Fpar_500m' in df.columns:
    alertas_rango.append({'chequeo': 'fpar_fuera_0_1', 'n_filas': int(((df['Fpar_500m'] < 0) | (df['Fpar_500m'] > 1)).sum())})

display(pd.DataFrame(alertas_rango))


## Consistencia del terreno

Las variables de terreno son estaticas por departamento. Por eso, para cada departamento deberia existir un unico valor por columna de terreno.


In [ ]:
consistencia_terreno = []
for col in terrain_cols:
    if col not in df.columns:
        continue
    uniques = df.groupby('departamento')[col].nunique(dropna=True).to_dict()
    consistencia_terreno.append({'variable': col, 'n_unicos_por_departamento': uniques})

display(pd.DataFrame(consistencia_terreno))


## Consistencia interna de fecha, mes y año

Como la base ya trae `mes` y `anio`, verificamos que coincidan con la fecha real parseada.


In [ ]:
df['mes_from_fecha'] = df['fecha'].dt.month
df['anio_from_fecha'] = df['fecha'].dt.year

incons_mes = int((df['mes'] != df['mes_from_fecha']).sum())
incons_anio = int((df['anio'] != df['anio_from_fecha']).sum())

print('Inconsistencias mes vs fecha:', incons_mes)
print('Inconsistencias anio vs fecha:', incons_anio)


## Decision de validacion

Con base en las revisiones anteriores, este bloque resume si la base puede pasar al siguiente paso del pipeline.


In [ ]:
decision = {
    'archivo_entrada': str(INPUT_PATH),
    'shape_base_original': tuple(pd.read_csv(INPUT_PATH).shape),
    'shape_base_trabajo': df.shape,
    'panel_completo': len(faltantes_panel) == 0 and len(sobrantes_panel) == 0,
    'duplicados_llave': duplicados_llave,
    'inconsistencias_mes_fecha': incons_mes,
    'inconsistencias_anio_fecha': incons_anio,
    'base_apta_para_siguiente_paso': (
        len(faltantes_panel) == 0 and
        duplicados_llave == 0 and
        incons_mes == 0 and
        incons_anio == 0
    ),
    'observaciones': [
        'los nulos deben interpretarse antes de imputar; no todos son errores estructurales',
        'si los faltantes se concentran al final de la serie, pueden corresponder a rezagos naturales de la fuente',
        'las variables de terreno deben permanecer estaticas por departamento',
    ],
}

print(json.dumps(decision, indent=2, ensure_ascii=False))


## Preparacion de la salida validada

La salida conserva la base original, pero ordenada y con columnas auxiliares removidas para no contaminar etapas posteriores.


In [ ]:
drop_aux = [c for c in ['mes_from_fecha', 'anio_from_fecha'] if c in df.columns]
df_validada = df.drop(columns=drop_aux).copy()

print('Shape de salida validada:', df_validada.shape)
display(df_validada.head(5))


## Validaciones finales antes de exportar

Estas validaciones garantizan que la copia validada no rompe la estructura esperada del pipeline.


In [ ]:
assert df_validada.duplicated(subset=['fecha', 'departamento']).sum() == 0, 'Hay duplicados por fecha-departamento.'
assert set(df_validada['departamento'].unique()) == {'Cundinamarca', 'Risaralda'}, 'Departamentos inesperados en la base clima-satelite.'
assert df_validada['fecha'].min() == pd.Timestamp('2000-01-01'), 'La fecha minima esperada es 2000-01-01.'
assert df_validada['fecha'].max() == pd.Timestamp('2026-02-01'), 'La fecha maxima esperada es 2026-02-01.'
assert len(faltantes_panel) == 0, 'Existen huecos estructurales en el panel mensual.'
assert incons_mes == 0, 'Existen inconsistencias entre mes y fecha.'
assert incons_anio == 0, 'Existen inconsistencias entre anio y fecha.'

print('Validaciones finales superadas correctamente.')


## Exportacion de la copia validada

Se guarda con separador `;` para mantener consistencia con las salidas intermedias del proyecto.


In [ ]:
df_validada.to_csv(OUTPUT_PATH, index=False, sep=';')
print(f'Archivo exportado en: {OUTPUT_PATH}')


## Resumen ejecutivo

Este bloque deja una lectura corta del resultado y sirve para enlazar con el siguiente notebook.


In [ ]:
resumen = {
    'archivo_salida': str(OUTPUT_PATH),
    'shape_validada': df_validada.shape,
    'periodo': f"{df_validada['fecha'].min().date()} a {df_validada['fecha'].max().date()}",
    'duplicados_llave': duplicados_llave,
    'panel_completo': len(faltantes_panel) == 0,
    'principales_nulos': nulos_por_columna.head(5).to_dict(orient='records'),
    'siguiente_paso': 'integrar esta base validada con la capa anual limpia de Agronet para construir la base mensual del seguro',
}

print(json.dumps(resumen, indent=2, ensure_ascii=False, default=str))


## Siguiente notebook recomendado

Con esta validacion cerrada, el siguiente paso natural es construir `04_integracion_base_mensual.ipynb` para unir esta base clima-satelite con la nueva capa anual de Agronet y preparar la capa mensual operativa del proyecto.
